In [15]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
# Use DATA_DIR from .env, or fallback to project root / data (run notebook from project root)
_data_dir = os.getenv("DATA_DIR")
if _data_dir is None:
    _data_dir = os.path.join(os.getcwd(), "data")
m5_accuracy_data_path = os.path.join(_data_dir, "m5-forecasting-accuracy")
sales_train_validation_df = pd.read_csv(os.path.join(m5_accuracy_data_path, "sales_train_validation.csv"))
# Working copy: use only `sales` from here on in this notebook
sales = sales_train_validation_df.copy()

# Reduce memory: downcast day columns (sales are small integers)
day_cols = [c for c in sales.columns if c.startswith("d_")]
sales[day_cols] = sales[day_cols].astype("int16")

# Use a sample to avoid kernel crash (melt + merge on full data use ~10GB+ RAM).
# Set to 1.0 for full data if you have enough memory.
SAMPLE_FRAC = 0.1
if SAMPLE_FRAC < 1.0:
    sales = sales.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

sales

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,0,0,0,0,...,0,0,1,2,0,0,0,0,1,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,2,0,2,0,...,0,2,0,1,0,0,0,0,0,1
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,0,0,0,0,...,0,0,1,0,0,1,1,2,0,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,0,0,0,0,...,0,1,1,2,2,0,2,2,1,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3044,HOBBIES_1_185_WI_1_validation,HOBBIES_1_185,HOBBIES_1,HOBBIES,WI_1,WI,1,1,0,0,...,0,1,0,1,0,0,0,0,0,0
3045,HOUSEHOLD_2_174_WI_3_validation,HOUSEHOLD_2_174,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
3046,HOUSEHOLD_1_177_CA_3_validation,HOUSEHOLD_1_177,HOUSEHOLD_1,HOUSEHOLD,CA_3,CA,0,0,6,2,...,0,0,0,0,1,0,0,0,0,0
3047,HOBBIES_1_129_WI_2_validation,HOBBIES_1_129,HOBBIES_1,HOBBIES,WI_2,WI,0,0,0,0,...,2,1,0,0,0,1,0,0,1,1


In [16]:
# Transform sales from wide (d_1, d_2, ...) to long format: one row per (id, day)
sales_long = sales.melt(
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='d',
    value_name='sales'
)
sales_long

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,d_1,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,d_1,2
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,d_1,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,d_1,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,d_1,0
...,...,...,...,...,...,...,...,...
5832732,HOBBIES_1_185_WI_1_validation,HOBBIES_1_185,HOBBIES_1,HOBBIES,WI_1,WI,d_1913,0
5832733,HOUSEHOLD_2_174_WI_3_validation,HOUSEHOLD_2_174,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1913,0
5832734,HOUSEHOLD_1_177_CA_3_validation,HOUSEHOLD_1_177,HOUSEHOLD_1,HOUSEHOLD,CA_3,CA,d_1913,0
5832735,HOBBIES_1_129_WI_2_validation,HOBBIES_1_129,HOBBIES_1,HOBBIES,WI_2,WI,d_1913,1


In [17]:
# Load calendar and merge with sales_long on day (d)
calendar = pd.read_csv(os.path.join(m5_accuracy_data_path, "calendar.csv"))
data = sales_long.merge(calendar, on='d', how='left')
data

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,d_1,2,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5832732,HOBBIES_1_185_WI_1_validation,HOBBIES_1_185,HOBBIES_1,HOBBIES,WI_1,WI,d_1913,0,2016-04-24,11613,...,2,4,2016,NaN,NaN,NaN,NaN,0,0,0
5832733,HOUSEHOLD_2_174_WI_3_validation,HOUSEHOLD_2_174,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1913,0,2016-04-24,11613,...,2,4,2016,NaN,NaN,NaN,NaN,0,0,0
5832734,HOUSEHOLD_1_177_CA_3_validation,HOUSEHOLD_1_177,HOUSEHOLD_1,HOUSEHOLD,CA_3,CA,d_1913,0,2016-04-24,11613,...,2,4,2016,NaN,NaN,NaN,NaN,0,0,0
5832735,HOBBIES_1_129_WI_2_validation,HOBBIES_1_129,HOBBIES_1,HOBBIES,WI_2,WI,d_1913,1,2016-04-24,11613,...,2,4,2016,NaN,NaN,NaN,NaN,0,0,0


In [18]:
# Load sell_prices and merge with data (join on store, item, and week)
# Drop sell_price from data if already present (avoids sell_price_x / sell_price_y when merge is run twice)
if 'sell_price' in data.columns:
    data = data.drop(columns=['sell_price'])
prices = pd.read_csv(os.path.join(m5_accuracy_data_path, "sell_prices.csv"))
data = data.merge(
    prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)
data

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,d_1,2,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,3.97
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5832732,HOBBIES_1_185_WI_1_validation,HOBBIES_1_185,HOBBIES_1,HOBBIES,WI_1,WI,d_1913,0,2016-04-24,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,3.24
5832733,HOUSEHOLD_2_174_WI_3_validation,HOUSEHOLD_2_174,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1913,0,2016-04-24,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,3.97
5832734,HOUSEHOLD_1_177_CA_3_validation,HOUSEHOLD_1_177,HOUSEHOLD_1,HOUSEHOLD,CA_3,CA,d_1913,0,2016-04-24,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,5.97
5832735,HOBBIES_1_129_WI_2_validation,HOBBIES_1_129,HOBBIES_1,HOBBIES,WI_2,WI,d_1913,1,2016-04-24,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,4.88


In [28]:
# Check if entire groups are missing price (all nulls in sell_price for that store-item)
n_groups_all_null = data.groupby(['store_id', 'item_id'])['sell_price'].apply(lambda x: x.isnull().all()).sum()
print(f"Number of (store_id, item_id) groups with entirely missing sell_price: {n_groups_all_null}")
n_groups_all_null

Number of (store_id, item_id) groups with entirely missing sell_price: 0


np.int64(0)

In [19]:
# Handle nulls in sell_price: sort by item-store and date, then forward fill within each group
# Sort so within each (store_id, item_id) group rows are chronological, then ffill
data = data.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)
data['sell_price'] = (
    data.groupby(['store_id', 'item_id'])['sell_price']
    .transform(lambda x: x.ffill())
)
data

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
1,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_2,9,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
2,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_3,3,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
3,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_4,3,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.94
4,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_5,0,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5832732,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1909,0,2016-04-20,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832733,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1910,0,2016-04-21,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832734,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1911,0,2016-04-22,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832735,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1912,0,2016-04-23,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94


In [20]:
# Number of rows in data
print(f"Number of rows in data: {len(data):,}")
len(data)

Number of rows in data: 5,832,737


5832737

In [21]:
# Load sell_prices and merge with data (join on store, item, and week)
# Drop sell_price from data if already present (avoids sell_price_x / sell_price_y when merge is run twice)
if 'sell_price' in data.columns:
    data = data.drop(columns=['sell_price'])
prices = pd.read_csv(os.path.join(m5_accuracy_data_path, "sell_prices.csv"))
data = data.merge(
    prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)
data

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
1,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_2,9,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
2,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_3,3,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
3,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_4,3,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.94
4,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_5,0,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5832732,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1909,0,2016-04-20,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832733,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1910,0,2016-04-21,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832734,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1911,0,2016-04-22,11612,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94
5832735,HOUSEHOLD_2_503_WI_3_validation,HOUSEHOLD_2_503,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1912,0,2016-04-23,11613,...,4,2016,NaN,NaN,NaN,NaN,0,0,0,6.94


In [22]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [23]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [24]:
data.isnull().sum()

id                    0
item_id               0
dept_id               0
cat_id                0
store_id              0
state_id              0
d                     0
sales                 0
date                  0
wm_yr_wk              0
weekday               0
wday                  0
month                 0
year                  0
event_name_1    5363191
event_type_1    5363191
event_name_2    5820541
event_type_2    5820541
snap_CA               0
snap_TX               0
snap_WI               0
sell_price      1239840
dtype: int64

In [25]:
data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
1,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_2,9,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
2,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_3,3,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94
3,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_4,3,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.94
4,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_5,0,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.94


In [26]:
data.shape
data.head()
data.tail()
data.describe()
data.info()
data.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5832737 entries, 0 to 5832736
Data columns (total 22 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            object 
 1   item_id       object 
 2   dept_id       object 
 3   cat_id        object 
 4   store_id      object 
 5   state_id      object 
 6   d             object 
 7   sales         int16  
 8   date          object 
 9   wm_yr_wk      int64  
 10  weekday       object 
 11  wday          int64  
 12  month         int64  
 13  year          int64  
 14  event_name_1  object 
 15  event_type_1  object 
 16  event_name_2  object 
 17  event_type_2  object 
 18  snap_CA       int64  
 19  snap_TX       int64  
 20  snap_WI       int64  
 21  sell_price    float64
dtypes: float64(1), int16(1), int64(7), object(13)
memory usage: 945.6+ MB


id                    0
item_id               0
dept_id               0
cat_id                0
store_id              0
state_id              0
d                     0
sales                 0
date                  0
wm_yr_wk              0
weekday               0
wday                  0
month                 0
year                  0
event_name_1    5363191
event_type_1    5363191
event_name_2    5820541
event_type_2    5820541
snap_CA               0
snap_TX               0
snap_WI               0
sell_price      1239840
dtype: int64